In [ ]:
                                                #  TEXT CLASSIFICATION

In [ ]:
!pip uninstall -y transformers datasets huggingface_hub tokenizers accelerate


In [ ]:
!pip install \
transformers==4.46.3 \
datasets==3.2.0 \
huggingface_hub==0.26.2 \
tokenizers==0.20.3 \
accelerate==1.1.1

In [ ]:
                                       # Using a Task-Specific Model

In [ ]:
# Imports the load_dataset() function from the Hugging Face datasets library.
from datasets import load_dataset

# Downloads and loads the Rotten Tomatoes sentiment dataset into the variable data.
data = load_dataset("rotten_tomatoes")
print(data)

In [ ]:
# retrieves the first and last examples from the training dataset.
data["train"][0, -1]

In [ ]:
# from transformers library importing pipeline
from transformers import pipeline


# Name (path) of the Hugging Face model to use
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"


# Load model into pipeline
pipe = pipeline(
   # Load this pre-trained model
model=model_path,
   # Load the matching tokenizer for the model
tokenizer=model_path,

   # return the prediction score for every possible class
return_all_scores=True,

   # Run the model on the CPU
device="cpu"
)

# Predict the sentiment of the given sentence
pipe("I hate this movie")

In [ ]:
import numpy as np

# tqdm = It creates a progress bar
from tqdm import tqdm

# Import KeyDataset, which lets the pipeline read only the review text from the dataset.
from transformers.pipelines.pt_utils import KeyDataset

# Create an empty list that will store all the model's predictions.
y_pred = []


# 1 Select only the review column from the test data set
# 2 run the sentiment model
# 3 show a progress bar
# 4 store each prediction in the variable output.
for output in tqdm(pipe(KeyDataset(data["test"], "text")),

                   # Len of the test dataset
                   total=len(data["test"])):

                   # Model preditcs the output as label and score .It predicts three labels for each review
                   # The index for
                   # 0 = negative
                   # 1 = neutral
                   # 2 = postive
                   # Store the negative sentiment score predicted by the model.
                   negative_score = output[0]["score"]

                   # STore the postive sentiment score predicted by the model
                   positive_score = output[2]["score"]

                   # Selecting the max score among those two
                   assignment = np.argmax([negative_score, positive_score])

                   # prediction is added to list
                   y_pred.append(assignment)

In [ ]:
# Imports the classification_report function to evaluate the classification model.
from sklearn.metrics import classification_report

# Creates a function that compares the actual labels with the predicted labels.
def evaluate_performance(y_true, y_pred):

  # classification_report() = Compares the actual labels (y_true) with the predicted labels (y_pred).
  performance = classification_report(
      y_true, y_pred,

      # Replaces numeric labels with readable names.
      target_names=["Negative Review", "Positive Review"]
)
  print(performance)

  # Inputs are revies from test data and y_predicted
evaluate_performance(data["test"]["label"], y_pred)

In [1]:
                   # Classification Tasks That Leverage Embeddings

In [ ]:
# It deletes the cached copy of the Hugging Face model sentence-transformers/all-mpnet-base-v2 from your local machine or Google Colab.
!rm -rf ~/.cache/huggingface/hub/models--sentence-transformers--all-mpnet-base-v2

In [ ]:
# It installs or upgrades three Python libraries to their latest versions.
!pip install -U sentence-transformers transformers accelerate

In [ ]:
# Import the class used to load pretrained embedding models
from sentence_transformers import SentenceTransformer

# Downloads the pretrained transformer that helps to convert into embeddings
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [ ]:
# model = Embedding model.
# .encode() = Converts text into embeddings.
# data["train"]["text"] = All training reviews.
# show_progress_bar=True  Show a progress bar while encoding.
train_embeddings = model.encode(data["train"]["text"],
show_progress_bar=True)

# Same for test data
test_embeddings = model.encode(data["test"]["text"],
show_progress_bar=True)
print(train_embeddings.shape)

In [ ]:
# Imports the Logistic Regression classifier.
from sklearn.linear_model import LogisticRegression

#clf → Variable name (classifier).
# random_state=42 = If you run the code again, you should get the same outcome.
clf = LogisticRegression(random_state=42)

# train the model with training embeddings which are review data in train dataset
clf.fit(train_embeddings, data["train"]["label"])

In [ ]:
# Import the function used to evaluate classification models
from sklearn.metrics import classification_report

# # Use the trained classifier to predict the labels of the test data
y_pred = clf.predict(test_embeddings)

# Function to evaluate the model's performance
def evaluate_performance(y_true, y_pred):
  """Create and print the classification report"""
  performance = classification_report(
      y_true, y_pred,
      target_names=["Negative Review", "Positive Review"]
)
  print(performance)

  evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
                                         # WITHOUT LABELLED DATA

In [ ]:
# Create embeddings for our labels
label_embeddings = model.encode(["A negative review",  "A positive review"])

In [ ]:
import numpy as np

# Imports the function that compares two embeddings.
from sklearn.metrics.pairwise import cosine_similarity

# Comapre every reviwe with every label
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)

# This chooses the largest similarity score in each row.
y_pred = np.argmax(sim_matrix, axis=1)

# Function call it prints the report
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
                                    # Using the Text-to-Text Transfer Transformer

In [ ]:
# Load our model
from transformers import pipeline

# Creates a pipeline.
pipe = pipeline(
"text2text-generation",
model="google/flan-t5-small",
device="cpu"
)

In [ ]:
# Creating the prompt
prompt = "Is the following sentence positive or negative? "

# adding every with the prompt and new column is created as t5 it stores prompt along with review
data = data.map(lambda example: {"t5": prompt + example['text']})

print(data)

In [ ]:
# Import tqdm to display a progress bar while processing the data
from tqdm import tqdm

# Import KeyDataset to read one column from the dataset efficiently
from transformers.pipelines.pt_utils import KeyDataset

# Which stores the predicted lables
y_pred = []

# Run the pipeline on each text in the "t5" column of the test dataset
for output in tqdm(pipe(KeyDataset(data["test"], "t5")),
total=len(data["test"])):

    # Get the generated text (either "negative" or "positive")
    text = output[0]["generated_text"]

    # Convert the text label into a numeric label
    # "negative" = 0
    # "positive" = 1
    y_pred.append(0 if text == "negative" else 1)

In [ ]:
                           # ChatGPT for Classification

In [ ]:
                           # USING THIS INSTEAD OF GPT

In [ ]:
from transformers import pipeline

# Create the FLAN-T5 pipeline
pipe = pipeline(
    "text2text-generation",   # Generate text from a text prompt
    model="google/flan-t5-small",  # model name
    device=-1   # run on CPU (-1), GPU (0)
)

In [ ]:
# Function to generate a response using the FLAN-T5 model
def chatgpt_generation(prompt, document, model="google/flan-t5-small"):
    """Generate an output based on a prompt and an input document."""

     # # Create a conversation with a system message and a user message
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",

            # Replace [DOCUMENT] in the prompt with the actual document
            "content": prompt.replace("[DOCUMENT]", document)
        }
    ]

    generation = pipe(
        messages[1]["content"],   # # Send only the user's prompt to FLAN-T5
        max_new_tokens=50,
        do_sample=False
    )

   # Return only the generated text
    return generation[0]["generated_text"]

In [ ]:
# Define a prompt template as a base
prompt = """Predict whether the following document is a positive
or negative movie review:
[DOCUMENT]
If it is positive return 1 and if it is negative return 0. Do not
give any other answers.
"""

In [ ]:
# Predict the target using GPT
document = "unpretentious , charming , quirky , original"

chatgpt_generation(prompt, document)

In [ ]:
# tore the predictions for all test reviews
predictions = [

    # Predict the sentiment for one review
    chatgpt_generation(prompt, doc)

    # Repeat for every review in the test dataset
    for doc in data["test"]["text"]
]

In [ ]:
# Extract predictions
# Convert the model's output into numeric labels
y_pred = [

# If the model returns "positive", store 1
    1 if pred.lower() == "positive"

# If the model returns "negative", store 0
    else 0 if pred.lower() == "negative"

# Otherwise, convert the returned value (0 or 1) to an intege
    else int(pred)

# Repeat for every prediction
    for pred in predictions
]

# y_pred = [int(pred) for pred in predictions]

In [ ]:
# Evaluate performance
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
# download and install the google-genai library.
pip install -U google-genai

In [ ]:
                                # Using Google GenAI for Classification instead of ChatGPT

In [ ]:
# # Import the Google GenAI library
from google import genai

# Create client by using api key
client = genai.Client(api_key="YOUR_API_KEY")

In [ ]:
# Function to generate a response using the Gemini model
def gemini_generation(prompt, document, model="models/gemini-flash-latest"):
    """Generate an output based on a prompt and an input document."""

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt.replace("[DOCUMENT]", document)
        }
    ]

    # Gemini expects the conversation as text
    conversation = ""

    # Combine all messages into a single text string
    for message in messages:
        conversation += f"{message['role']}: {message['content']}\n"

    # Send the conversation to the Gemini model
    response = client.models.generate_content(
        model=model,
        contents=conversation     # Input text sent to the model
    )

 # Return only the generated response
    return response.text.strip()

In [ ]:
# Prompt
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0.
Do not give any other answers.
"""

# Predict
# Example movie review to classify
document = "unpretentious, charming, quirky, original"


# # Predict the sentiment using Gemini and print the result
print(gemini_generation(prompt, document))


In [ ]:
      #  PREDICTIONS ARE NOT POSSIBILE WITH THE ABOVE